# Avance Fase 3 — Semana 2
## Núcleo algorítmico y Programación Orientada a Objetos (versión guía)

**Curso:** MCDI500 — Programación para la Ciencia de Datos · **Programa:** Magíster en Ciencia de Datos e IA, UNAB
**Proyecto de ejemplo:** Predicción de accidente cerebrovascular (*stroke*)
**Integrantes:** _(completar)_ · **Docente:** Omar Salinas Silva · **Fecha:** _(completar)_

---

> **¿Para qué sirve este notebook?**
> Es una **guía paso a paso** para llevar tu pipeline de la Fase 2 (hecho con *funciones*) hacia
> **Programación Orientada a Objetos (POO)**. Está pensado para quienes **recién empiezan** con POO:
> avanza despacio, con ejemplos simples y mucha explicación. Al final hay una sección sobre **cómo
> adaptarlo a tu propio proyecto**.
>
> Para usar tus datos reales, coloca tu CSV junto al notebook y ejecuta **Kernel → Restart & Run All**.


## Contenido

1. ¿Por qué POO? De funciones a objetos
2. Los 5 conceptos que necesitas (en 1 minuto)
3. Tu primera clase: `Preprocesador`
4. Usar la clase paso a paso
5. Herencia: reutilizar y especializar
6. Recursividad (idea simple)
7. Eficiencia: ¿por qué evitar bucles?
8. Cómo aplicar esto a **tu** proyecto
9. Bibliografía


## 1. ¿Por qué POO? De funciones a objetos

En la **Fase 2** resolviste el preprocesamiento con **funciones**: `cargar_datos()`, `limpiar()`,
`codificar()`… Cada función recibía el `DataFrame`, lo modificaba y lo devolvía. Funciona, pero los
datos (`df`) y los pasos andaban **sueltos** por el notebook, y era fácil perder el hilo de en qué
estado estaba la tabla.

La **idea de la POO** es juntar en un mismo lugar **los datos y las acciones que operan sobre ellos**.
Ese "lugar" es un **objeto**, creado a partir de una **clase** (un molde).

Piensa en el ejemplo de la `Moto` que viste en la Semana 2:

```python
class Moto:
    def __init__(self, modelo):
        self.modelo = modelo      # un ATRIBUTO (dato que vive dentro del objeto)
    def arrancar(self):           # un METODO (accion del objeto)
        print(self.modelo, "arrancando...")
```

Aquí haremos lo mismo, pero en vez de una moto, nuestro objeto será un **Preprocesador**: un objeto
que **guarda tu tabla de datos adentro** y sabe limpiarla, codificarla y escalarla.


## 2. Los 5 conceptos que necesitas (en 1 minuto)

| Concepto | Qué es | En nuestro ejemplo |
|---|---|---|
| **Clase** | El molde o plantilla. | `class Preprocesador:` |
| **Objeto** | Algo creado a partir de la clase. | `prep = Preprocesador("datos.csv")` |
| **Atributo** | Un dato que vive dentro del objeto. | `self.df` (la tabla) |
| **Método** | Una acción del objeto (una función dentro de la clase). | `prep.limpiar()` |
| **`self`** | La palabra que usa el objeto para referirse **a sí mismo**. | `self.df = ...` |

Y un método especial: **`__init__`** es el **constructor**: se ejecuta automáticamente cuando creas el
objeto y sirve para dejar listos sus atributos iniciales.


## 3. Tu primera clase: `Preprocesador`

Vamos a meter **todo el pipeline de la Fase 2 dentro de una sola clase**. Cada paso que antes era una
función, ahora será un **método**. La gran diferencia: la tabla se guarda en el atributo `self.df`, así
que ya no tenemos que ir pasándola de función en función.

Primero, una pequeña ayuda para que el notebook **siempre se pueda ejecutar** aunque no tengas el CSV a
mano: si no encuentra el archivo real, genera datos de prueba con el mismo formato.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler

%matplotlib inline
np.random.seed(42)   # reproducibilidad: mismos resultados cada vez


def generar_datos_sinteticos(n=5110, semilla=42):
    """Crea datos de PRUEBA con el mismo formato del dataset de stroke.
    Solo se usa si no encuentras el CSV real, para que el notebook corra igual."""
    rng = np.random.default_rng(semilla)
    df = pd.DataFrame({
        "id": rng.choice(np.arange(1, n * 20), size=n, replace=False),
        "gender": rng.choice(["Male", "Female", "Other"], size=n, p=[0.41, 0.585, 0.005]),
        "age": np.round(rng.uniform(0.08, 82, size=n), 1),
        "hypertension": rng.choice([0, 1], size=n, p=[0.9, 0.1]),
        "heart_disease": rng.choice([0, 1], size=n, p=[0.95, 0.05]),
        "ever_married": rng.choice(["Yes", "No"], size=n, p=[0.66, 0.34]),
        "work_type": rng.choice(["Private", "Self-employed", "Govt_job", "children", "Never_worked"],
                                size=n, p=[0.57, 0.16, 0.13, 0.13, 0.01]),
        "Residence_type": rng.choice(["Urban", "Rural"], size=n, p=[0.51, 0.49]),
        "avg_glucose_level": np.round(rng.uniform(55, 272, size=n), 2),
        "bmi": np.round(rng.normal(28, 7, size=n), 1),
        "smoking_status": rng.choice(["never smoked", "Unknown", "formerly smoked", "smokes"],
                                     size=n, p=[0.37, 0.30, 0.17, 0.16]),
        "stroke": rng.choice([0, 1], size=n, p=[0.951, 0.049]),
    })
    idx = rng.choice(n, size=int(0.039 * n), replace=False)
    df.loc[idx, "bmi"] = np.nan
    return df

Ahora **la clase**. Léela con calma: cada método está comentado y hace **una sola cosa**.
Fíjate cómo todos usan `self.df` (la tabla que vive dentro del objeto).


In [ ]:
class Preprocesador:
    """Objeto que guarda una tabla de datos y sabe prepararla para el analisis.

    La tabla vive en el atributo self.df. Cada metodo (cargar, limpiar, codificar,
    escalar, validar) es un paso del pipeline que trabaja sobre esa misma tabla.
    """

    def __init__(self, ruta):
        # __init__ es el constructor: deja listos los atributos del objeto.
        self.ruta = ruta        # donde esta el archivo de datos
        self.df = None          # aqui guardaremos la tabla (todavia vacia)
        self.scaler = None      # aqui guardaremos el escalador (para reusarlo luego)

    def cargar(self):
        """Lee el CSV y lo guarda en self.df. Si no existe, usa datos de prueba."""
        if os.path.exists(self.ruta):
            self.df = pd.read_csv(self.ruta)
        else:
            self.df = generar_datos_sinteticos()
            print("Aviso: no encontre el CSV real, uso datos de prueba.")
        print(f"Datos cargados: {self.df.shape[0]} filas, {self.df.shape[1]} columnas")
        return self.df

    def explorar(self):
        """Muestra el estado inicial: dimensiones y cuantos nulos hay."""
        print("Dimensiones:", self.df.shape)
        print("\nNulos por columna:")
        print(self.df.isnull().sum())
        return self.df.describe()

    def limpiar(self):
        """Quita la columna 'id' y rellena los nulos de 'bmi' con la mediana."""
        if "id" in self.df.columns:
            self.df = self.df.drop(columns=["id"])
        n = int(self.df["bmi"].isnull().sum())
        mediana = self.df["bmi"].median()              # mediana: robusta a valores extremos
        self.df["bmi"] = self.df["bmi"].fillna(mediana)
        print(f"Eliminada 'id'. bmi: {n} nulos rellenados con la mediana ({mediana:.1f}).")
        return self.df

    def codificar(self, columna, nombres):
        """Convierte una columna de texto en columnas 0/1 (One-Hot)."""
        le = preprocessing.LabelEncoder().fit(self.df[columna])
        if len(nombres) != len(le.classes_):
            raise ValueError(
                f"'{columna}' tiene {len(le.classes_)} categorias {list(le.classes_)}, "
                f"pero diste {len(nombres)} nombres.")
        codigos = le.transform(self.df[columna]).reshape(-1, 1)
        ohe = preprocessing.OneHotEncoder().fit(codigos)
        matriz = ohe.transform(codigos).toarray()
        nuevas = pd.DataFrame(matriz, columns=nombres, index=self.df.index).astype(int)
        self.df = pd.concat([self.df.drop(columns=[columna]), nuevas], axis=1)
        print(f"'{columna}' -> {nombres}")
        return self.df

    def escalar(self, columnas):
        """Estandariza las columnas continuas (las deja con media 0 y desviacion 1)."""
        self.scaler = StandardScaler().fit(self.df[columnas])
        self.df[columnas] = self.scaler.transform(self.df[columnas])
        print(f"Escaladas: {columnas}")
        return self.df

    def validar(self):
        """Revisa que la tabla quedo sin nulos y sin texto sin codificar."""
        nulos = int(self.df.isnull().sum().sum())
        texto = self.df.select_dtypes(exclude=[np.number]).columns.tolist()
        print("Nulos totales:", nulos)
        print("Columnas de texto sin codificar:", texto if texto else "ninguna")
        print("Dimensiones finales:", self.df.shape)
        return nulos == 0 and not texto

## 4. Usar la clase paso a paso

Creamos **un objeto** de la clase y vamos llamando sus métodos en orden. Observa lo cómodo que es: no
pasamos `df` por ningún lado, porque la tabla ya vive **dentro del objeto** `prep`.


In [ ]:
# 1) Creamos el objeto (esto ejecuta __init__)
prep = Preprocesador("healthcare-dataset-stroke-data.csv")

# 2) Cargamos y miramos los datos
prep.cargar()
prep.explorar()

In [ ]:
# 3) Limpiamos (quitar id + imputar bmi)
prep.limpiar()

# 4) Codificamos las 5 variables de texto (One-Hot)
prep.codificar("gender", ["gender_Female", "gender_Male", "gender_Other"])
prep.codificar("ever_married", ["married_No", "married_Yes"])
prep.codificar("work_type", ["work_Govt", "work_Never", "work_Private", "work_Self", "work_children"])
prep.codificar("Residence_type", ["res_Rural", "res_Urban"])
prep.codificar("smoking_status", ["smoke_Unknown", "smoke_formerly", "smoke_never", "smoke_smokes"])

# 5) Escalamos las variables continuas
prep.escalar(["age", "avg_glucose_level", "bmi"])

# 6) Validamos el resultado
print("\n¿Tabla lista para el analisis?:", prep.validar())

In [ ]:
# La tabla final vive en prep.df
prep.df.head()

In [ ]:
# Grafico de control: las continuas quedaron centradas en ~0 y en escala parecida
prep.df[["age", "avg_glucose_level", "bmi"]].boxplot(figsize=(9, 4))
plt.title("Variables continuas estandarizadas")
plt.show()

## 5. Herencia: reutilizar y especializar

La **herencia** permite crear una clase nueva **a partir de otra**, reutilizando lo que ya existe y
**cambiando solo lo necesario**. La clase de la que se hereda es la **clase padre (base)** y la nueva
es la **clase hija**.

Ejemplo sencillo: una clase `Limpiador` que limpia eliminando filas con nulos, y una hija
`LimpiadorPorMediana` que hereda de ella pero **cambia la forma de limpiar** (rellena con la mediana en
vez de eliminar). Nota que el método se llama **igual** (`limpiar`) pero hace algo distinto: a eso se
le llama **polimorfismo**.


In [ ]:
class Limpiador:
    """Clase base: limpia eliminando las filas que tengan algun nulo."""
    def limpiar(self, df):
        return df.dropna()


class LimpiadorPorMediana(Limpiador):       # <- hereda de Limpiador
    """Clase hija: en vez de eliminar filas, rellena los nulos con la mediana."""
    def limpiar(self, df):                  # mismo nombre, comportamiento distinto (polimorfismo)
        df = df.copy()
        for columna in df.select_dtypes(include="number").columns:
            df[columna] = df[columna].fillna(df[columna].median())
        return df


# Probamos ambas sobre una mini-tabla con nulos
mini = pd.DataFrame({"a": [1.0, np.nan, 3.0], "b": [4.0, 5.0, np.nan]})
print("Tabla original:", mini.shape, "filas x columnas")
print("Con Limpiador (elimina filas):     ", Limpiador().limpiar(mini).shape)
print("Con LimpiadorPorMediana (rellena): ", LimpiadorPorMediana().limpiar(mini).shape,
      "->", int(LimpiadorPorMediana().limpiar(mini).isnull().sum().sum()), "nulos")

**¿Para qué sirve esto en tu proyecto?** Podrías tener una clase base con la limpieza común y
clases hijas con estrategias distintas (por mediana, por media, por eliminación), y elegir cuál usar
sin reescribir todo. Por ahora, basta con que entiendas la idea: **heredar = reutilizar + especializar**.


## 6. Recursividad (idea simple)

Una función **recursiva** es una función que **se llama a sí misma** para resolver un problema más
pequeño, hasta llegar a un **caso base** que la detiene. Si no hay caso base, ¡se llama para siempre!

Dos ejemplos cortos:


In [ ]:
def cuenta_regresiva(n):
    """Imprime de n hasta 1 y luego se detiene."""
    if n == 0:                 # CASO BASE: aqui se detiene
        print("Despegue!")
        return
    print(n)
    cuenta_regresiva(n - 1)    # CASO RECURSIVO: se llama con un numero mas chico


def suma_lista(numeros):
    """Suma todos los elementos de una lista, de forma recursiva."""
    if len(numeros) == 0:      # CASO BASE: lista vacia suma 0
        return 0
    return numeros[0] + suma_lista(numeros[1:])   # primero + suma del resto


cuenta_regresiva(5)
print("Suma recursiva de [1,2,3,4,5]:", suma_lista([1, 2, 3, 4, 5]))

La clave de toda recursión: **(1) un caso base** que la detiene y **(2) un caso recursivo** que
avanza hacia ese caso base. En ciencia de datos la recursión aparece, por ejemplo, al recorrer
estructuras anidadas (carpetas, árboles de decisión). Para tu proyecto, basta con que puedas mostrar un
ejemplo recursivo simple y bien explicado como estos.


## 7. Eficiencia: ¿por qué evitar bucles?

Cuando los datos crecen, **cómo** escribimos el código importa. Comparemos dos formas de hacer lo
mismo —duplicar y sumar un millón de números— y midamos el tiempo con `timeit`:

- **Con un bucle de Python:** recorre número por número (lento).
- **Vectorizado con NumPy:** opera sobre todo el arreglo de una vez (mucho más rápido).


In [ ]:
import timeit

n = 1_000_000
lista = list(range(n))      # para el bucle
arreglo = np.arange(n)      # para NumPy


def con_bucle():
    total = 0
    for x in lista:         # recorre uno por uno
        total += x * 2
    return total


def vectorizado():
    return int((arreglo * 2).sum())   # una sola operacion sobre todo el arreglo


# Verificamos que dan el mismo resultado, y luego medimos
assert con_bucle() == vectorizado()
t_bucle = timeit.timeit(con_bucle, number=10)
t_vec = timeit.timeit(vectorizado, number=10)

print(f"Con bucle:    {t_bucle:.4f} s")
print(f"Vectorizado:  {t_vec:.4f} s")
print(f"El vectorizado fue ~{t_bucle / t_vec:.0f} veces mas rapido.")

**Idea para llevarte (sin matemática complicada):** mientras más datos, más se nota la
diferencia. Por eso en el `Preprocesador` usamos operaciones de pandas/NumPy en lugar de bucles
manuales. Esta forma de "medir para decidir" es justo lo que pide la rúbrica cuando habla de
*eficiencia y complejidad*.


## 8. Cómo aplicar esto a **tu** proyecto

Para reutilizar este notebook con **tu propio dataset**, sigue estos pasos:

1. **Cambia la ruta** del CSV al crear el objeto: `prep = Preprocesador("mi_dataset.csv")`.
2. **Ajusta `limpiar()`**: cambia el nombre de las columnas a eliminar y la(s) columna(s) numérica(s)
   con nulos que quieras imputar.
3. **Ajusta las llamadas a `codificar()`**: pon **tus** columnas de texto y, para cada una, una lista de
   nombres del **mismo largo** que la cantidad de categorías (si te equivocas en el número, la clase te
   avisa con un error claro).
4. **Ajusta `escalar()`**: indica **tus** columnas continuas (no escales binarias ni la variable
   objetivo).
5. Ejecuta `prep.validar()` para confirmar que quedó sin nulos y sin texto.
6. Vuelve a ejecutar todo con **Kernel → Restart & Run All** para que queden las salidas reales.

> Consejo: empieza simple (esta clase basta para aprobar la parte de POO). Cuando te sientas cómodo,
> puedes investigar pasos siguientes como separar cada transformación en su propia clase o usar
> clases abstractas; pero **no es necesario para partir**.


## 9. Bibliografía (APA 7)

> Reemplaza las entradas _(material del curso)_ por las de tu asignatura. La rúbrica pide al menos
> 5 fuentes (≥ 2 docentes, ≥ 2 documentación técnica, ≥ 1 académica).

- The pandas development team. (2024). *pandas documentation*. https://pandas.pydata.org/docs/
- Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.
- McKinney, W. (2022). *Python for data analysis* (3rd ed.). O'Reilly Media.
- Fedesoriano. (2021). *Stroke prediction dataset* [Conjunto de datos]. Kaggle. https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset
- Salinas, O. (2026). *Material de la Semana 2: desarrollo algorítmico y POO* [Material del curso]. MCDI500, UNAB. _(completar)_
- Universidad Andrés Bello. (2026). *Guía de la Fase 3* [Material del curso]. MCDI500. _(completar)_
